# Dataset Preprocessing Notebook



## 1. Mount Drive
Mount Google Drive, import libraries, and define dataset paths.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Import Libraries
Standard libraries for file handling, data manipulation, image processing, and visualisation.

OpenCV (`cv2`) is required for inpainting operations.

Pillow (`PIL`) is used for image loading and colour handling.

In [ ]:
import os
import hashlib
from pathlib import Path
import cv2
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import sys

### Dependencies
Installing any libraries not available by default in Colab.

In [ ]:
!pip install openpyxl -q
!pip install easyocr -q

## 3. Define Paths
All dataset paths are defined relative to the thesis root directory on Google Drive.

Centralising paths here means a single update propagates throughout the entire notebook if the folder structure changes.

In [ ]:
from os.path import exists

# Root path
ROOT = Path('/content/drive/MyDrive/University/Masters/MSc_Thesis')

# Raw data paths
RAW = ROOT / 'data' / 'raw'

BUSI_RAW = RAW / 'BUSI' / 'Dataset_BUSI' / 'Dataset_BUSI_with_GT'
BUSUCLM_RAW = RAW / 'BUS-UCLM' / 'BUS-UCLM Breast ultrasound lesion segmentation dataset' / 'BUS-UCLM'
BREAST_RAW = RAW / 'BrEaST' / 'BrEaST-Lesions' / 'BrEaST-Lesions_USG-images_and_masks'
BREAST_META = RAW / 'BrEaST'

# Outputs
OUTPUTS = ROOT / 'outputs'
RESULTS = OUTPUTS / 'results'
LOGS = OUTPUTS / 'logs'

# Source files
SRC = ROOT / 'src'
DECISIONS_CSV = SRC / 'decisions_manual.csv'
CRITIQUE_CSV = SRC / 'dataset_comment_list.csv'
AUDIT_CSV = RESULTS / 'audit_master.csv'

# Processed output
PROCESSED = ROOT / 'data' / 'processed'
BUSI_PROCESSED = PROCESSED / 'BUSI'
BUSUCLM_PROCESSED = PROCESSED / 'BUS-UCLM'
BREAST_PROCESSED = PROCESSED / 'BrEaST'

# Final output
FINAL = ROOT / 'data' / 'final'
BUSI_FINAL = FINAL / 'BUSI'
BUSUCLM_FINAL = FINAL / 'BUS-UCLM'
BREAST_FINAL = FINAL / 'BrEaST'

# Logs
LOGS = OUTPUTS / 'logs'

# Labels
LABELS = ['benign', 'malignant', 'normal']

# Verify critical paths
print('Path verification: ')
for name, path in [
    ('BUSI raw', BUSI_RAW),
    ('BUS-UCLM raw', BUSUCLM_RAW),
    ('BrEaST raw', BREAST_RAW),
    ('Decisions CSV', DECISIONS_CSV),
    ('Critique CSV', CRITIQUE_CSV),
    ('Audit master CSV', AUDIT_CSV),
    ('Results folder', RESULTS),
    ('Logs folder', LOGS)
]:
  print(f' {name}: {"Found" if path.exists() else "NOT FOUND"}')

## 4. Loading Source Data

Loading audit master dataframe and both decision (mine and BrEaST authors' critique of the BUSI dataset) CSV files.

All exclusion logic operates on these three sources. No other data is loaded at this stage.

In [ ]:
# Audit master dataframe from Notebook 01
df = pd.read_csv(AUDIT_CSV)
print(f'Audit master loaded: {len(df)} images')

# Manual decisions (radiologist + visual inspection)
decisions = pd.read_csv(DECISIONS_CSV)
print(f'Manual decisions loaded: {len(decisions)} rows')
print(f' exclude: {(decisions["action"] == "exclude").sum()}')
print(f' keep {(decisions["action"] == "keep").sum()}')

# BrEaST authors' critique CSV (axilla, needle, near-duplicates)
critique = pd.read_csv(CRITIQUE_CSV, delimiter=';')
print(f'Critique CSV loaded: {len(critique)} rows')
print(f' Objection values: {critique["Objection"].value_counts().to_dict()}')

## 5. Build Exclusion Set

Constructs the master exclusion set from two sources:
1. BrEaST authors' critique CSV (axilla and biopsy needle images)
2. Manual decisions CSV (radiologist + visual inspection)

Multiclass entries in the critique CSV are resolved by the radiologist decisions CSV and are not processed here.

The exclusion set contains filenames only. Matching against the master dataframe happens later.

In [ ]:
def parse_critique_filenames(filename_cell):

  cleaned = filename_cell.strip('{}')
  names = [name.strip() for name in cleaned.split('&')]
  return [f'{name}.png' for name in names]

# Source 1: critique CSV (axilla and biopsy needle images)
axilla_needle = critique[
    critique['Objection'].isin(['axilla', 'needle'])
].copy()

critique_exclusions = set()
for _, row in axilla_needle.iterrows():
  filenames = parse_critique_filenames(row['Filename'])
  critique_exclusions.update(filenames)

print(f'Critique CSV exclusions (axilla + biopsy needle): {len(critique_exclusions)} images')

# Source 2: manual decisions CSV (radiologists decisions on multiclass images and visual inspection of Doppler signal)
manual_exclusions = set(
    decisions[decisions['action'] == 'exclude']['filename'] + '.png'
)

print(f'Manual decision exclusions: {len(manual_exclusions)} images')

# Combining into master exclusion set
all_exclusions = critique_exclusions | manual_exclusions
print(f'Total unique exclusions: {len(all_exclusions)} images')

# Overlap check
overlap = critique_exclusions & manual_exclusions
if overlap:
  print(f'\nNote: {len(overlap)} images appear in both sources:')
  for name in sorted(overlap):
    print(f' {name}')
else:
  print(f'\nNo overlap between sources -- exclusion sets are clean.')

### Near-Duplicate Deduplication

BrEaST authors' critique CSV also encodes near-duplicate groups: rows with no `Objection` flag whose `Filename` field lists multiple `&`-joined images.

Tie-break rule: within each group, the image with the larger pixel area (width × height) is kept; all other members are excluded.

In [ ]:
dedup_candidates = critique[
    critique['Objection'].isna() &
    critique['Filename'].str.contains('&', regex=False)
].copy()

print(f'Rows with no Objection flag: {critique["Objection"].isna().sum()}')
print(f'Of those, rows that are actual duplicate groups (contain "&"): {len(dedup_candidates)}')

dedup_candidates['group_members'] = dedup_candidates['Filename'].apply(parse_critique_filenames)
dedup_candidates['group_size'] = dedup_candidates['group_members'].apply(len)

print('\nGroup size distribution:')
print(dedup_candidates['group_size'].value_counts().sort_index().to_string())

In [ ]:
busi_lookup = df[df['dataset'] == 'BUSI'].set_index('image_name')
resolution_lookup = busi_lookup['width'] * busi_lookup['height']

excluded_elsewhere = critique_exclusions | manual_exclusions

dedup_exclusions = set()
ties = []
missing_from_audit = []
fully_preexcluded_groups = []
single_eligible_groups = []

for _, row in dedup_candidates.iterrows():
  members = row['group_members']

  found = [m for m in members if m in resolution_lookup.index]
  not_found = [m for m in members if m not in resolution_lookup.index]
  missing_from_audit.extend(not_found)

  if len(found) < 2:
    continue

  # Exclude anything already removed by axilla/needle/manual decisions from the
  # comparison pool
  eligible = [m for m in found if m not in excluded_elsewhere]

  if len(eligible) == 0:
    fully_preexcluded_groups.append({'ID': row['ID'], 'members': found})
    continue

  if len(eligible) == 1:
    single_eligible_groups.append({
        'ID': row['ID'],
        'kept_by_elimination': eligible[0],
        'excluded_elsewhere': [m for m in found if m != eligible[0]]
    })
    keep = eligible[0]
  else:
    # Tie-break rule: keep the image with the larger pixel area (width x height)
    areas = {m: resolution_lookup[m] for m in eligible}
    max_area = max(areas.values())
    winners = [m for m, a in areas.items() if a == max_area]

    if len(winners) > 1:
      ties.append(areas)
      # Deterministic fallback for exact ties
      keep = sorted(winners)[0]
    else:
      keep = winners[0]

  dedup_exclusions.update(m for m in eligible if m != keep)

print(f'Duplicate groups processed: {len(dedup_candidates)}')
print(f'Images excluded via dedup: {len(dedup_exclusions)}')
print(f'Exact pixel-area ties encountered: {len(ties)}')
print(f'Groups fully pre-excluded by other means: {len(fully_preexcluded_groups)}')
print(f'Groups decided by elimination, not area: {len(single_eligible_groups)}')
if ties:
  print('\nTied groups (resolved alphabetically -- review by eye):')
  for areas in ties:
    print(f'  {areas}')
if fully_preexcluded_groups:
  print(f'\nGroups where every member was already excluded elsewhere:')
  for g in fully_preexcluded_groups:
    print(f'  {g}')
if single_eligible_groups:
  print('\nGroups decided by elimination, not area comparison:')
  for g in single_eligible_groups:
    print(f'  {g}')
if missing_from_audit:
  print(f'\nWARNING: {len(missing_from_audit)} critique filenames not found in audit master:')
  print(f'  {missing_from_audit}')

In [ ]:
# Structural guarantee: dedup can never exclude an image already excluded elsewhere
assert dedup_exclusions.isdisjoint(excluded_elsewhere), (
  'dedup_exclusions overlaps with critique/manual exclusions -- '
  'should be structurally impossible now. Investigate before proceeding.'
)

overlap_with_critique = dedup_exclusions & critique_exclusions
overlap_with_manual = dedup_exclusions & manual_exclusions

if overlap_with_critique or overlap_with_manual:
  print('WARNING: dedup exclusions overlap with existing exclusion sources.')
  print(f'  vs axilla/needle: {len(overlap_with_critique)}')
  print(f'  vs manual: {len(overlap_with_manual)}')
else:
  print('No overlap between dedup exclusions and existing sources -- clean.')

all_exclusions = critique_exclusions | manual_exclusions | dedup_exclusions
print(f'\nTotal unique exclusions (axilla/needle + manual + dedup): {len(all_exclusions)} images.')

## 6. Apply Exclusions to Master Dataframe

Filters the audit master dataframe by removing all images in the exclusion set. Operates on BUSI only because the remaining datasets are validation-only datasets with no training exclusions at this stage.

Post-exclusion counts are verified against expected values before proceeding.

In [ ]:
busi_mask = (
    (df['dataset'] == 'BUSI') &
    (df['image_name'].isin(all_exclusions))
)

n_before = (df['dataset'] == 'BUSI').sum()
df_clean = df[~busi_mask].copy()
n_after = (df_clean['dataset'] == 'BUSI').sum()
n_removed = n_before - n_after

print(f'BUSI images before exclusion: {n_before}')
print(f'BUSI images excluded: {n_removed}')
print(f'BUSI images after exclusion: {n_after}')

print('\nPost-exclusion class distribution (BUSI):')
busi_clean = df_clean[df_clean['dataset'] == 'BUSI']
print(busi_clean.groupby('class')['image_name'].count().to_string())

remaining_excluded = busi_clean[
    busi_clean['image_name'].isin(all_exclusions)
]
if len(remaining_excluded) == 0:
  print(f'\nVerification passed: no excluded images remain in BUSI.')
else:
  print(f'\n{len(remaining_excluded)} excluded images still present.')
  print(remaining_excluded[['image_name', 'class']].to_string())

print('\nFull post-exclusion dataset summary:')
print(df_clean.groupby(['dataset', 'class'])['image_name'].count().to_string())

## 7. BUS-UCLM Dataset Exclusions

Two exclusion types apply to BUS-UCLM:
1. Combined images: two views in one frame, no principled way to select a single representative view.
2. Doppler images.

BUS-UCLM is blind external validation only. These exclusions preserve signal consistency with the training distribution.

In [ ]:
busuclm_mask = df_clean['dataset'] == 'BUS-UCLM'
busuclm_df = df_clean[busuclm_mask].copy()

n_before = len(busuclm_df)

# Case- and whitespace-insensitive: raw metadata contains both 'Yes' and 'Y'
combined_mask = busuclm_df['Combined'].astype(str).str.strip().str.lower().isin(['yes', 'y'])
combined_filenames = set(busuclm_df[combined_mask]['image_name'])
n_combined = len(combined_filenames)

print(f'Combined images identified: {n_combined}')
print('  ' + ', '.join(sorted(combined_filenames)))

# Same tolerant matching applied to the Doppler flag
doppler_mask = busuclm_df['Doppler'].astype(str).str.strip().str.lower().isin(['yes', 'y'])
doppler_filenames = set(busuclm_df[doppler_mask]['image_name'])
n_doppler = len(doppler_filenames)

print(f'\nDoppler images identified: {n_doppler}')
print('By class:')
print(busuclm_df[doppler_mask].groupby('class')['image_name'].count().to_string())

busuclm_exclusions = combined_filenames | doppler_filenames
overlap = combined_filenames & doppler_filenames
if overlap:
  print(f'\nNote: {len(overlap)} images flagged as both Combined and Doppler.')

busuclm_exclude_mask = (
  (df_clean['dataset'] == 'BUS-UCLM') &
  (df_clean['image_name'].isin(busuclm_exclusions))
)

df_clean = df_clean[~busuclm_exclude_mask].copy()
n_after = (df_clean['dataset'] == 'BUS-UCLM').sum()

print(f'\nBUS-UCLM images before exclusion: {n_before}')
print(f'BUS-UCLM images excluded:         {n_before - n_after}')
print(f'BUS-UCLM images after exclusion:  {n_after}')

print('\nPost-exclusion class distribution (BUS-UCLM):')
print(df_clean[df_clean['dataset'] == 'BUS-UCLM'].groupby(
  'class')['image_name'].count().to_string())

### BUS-UCLM Exclusion Correction

Image `STSP_026.png` was flagged as Combined in `INFO.csv` but visual inspection confirmed no split view is present. The image is a standard B-Mode acquisition with a visible benign lesion, consistent with its class label. The Combined flag is a metadata annotation error.

This correction is applied before any further processing.

In [ ]:
correction_filename = 'STSP_026.png'

currently_in = correction_filename in df_clean['image_name'].values
print(f'STSP_026.png currently in df_clean: {currently_in}')

correction_row = df[df['image_name'] == correction_filename].copy()

if len(correction_row) == 0:
  print('Image not found in audit master.')
elif currently_in:
  print('Image already present in df_clean -- no action needed.')
else:
  df_clean = pd.concat([df_clean, correction_row], ignore_index = True)
  print(f'STSP_026.png reinstated.')
  print(f'BUS-UCLM count after correction: '
        f'{(df_clean["dataset"] == "BUS-UCLM").sum()}')

### Doppler Flag Correction

`COPE_018.png` is flagged `Doppler = Yes` in `INFO.csv`. Visual inspection found no Doppler signal present, therefore, the image has been reinstated.

In [ ]:
doppler_correction_filename = 'COPE_018.png'

currently_in = doppler_correction_filename in df_clean['image_name'].values
print(f'COPE_018.png currently in df_clean: {currently_in}')

correction_row = df[df['image_name'] == doppler_correction_filename].copy()

if len(correction_row) == 0:
  print('Image not found in audit master.')
elif currently_in:
  print('Image already present in df_clean -- no action needed.')
else:
  df_clean = pd.concat([df_clean, correction_row], ignore_index = True)
  print(f'COPE_018.png reinstated.')
  print(f'BUS-UCLM count after correction: '
        f'{(df_clean["dataset"] == "BUS-UCLM").sum()}')

### Manual Doppler Exclusion

`SIBA_004.png` contains a visible Doppler signal not flagged in `INFO.csv`. After visual check of the entire dataset and cross-reference with the `INFO.csv`, it was determined that only `SIBA_004.png` has been mislabeled. Therefore, the image has been excluded.

In [ ]:
busuclm_manual_exclusions = {'SIBA_004.png'}

busuclm_manual_mask = (
    (df_clean['dataset'] == 'BUS-UCLM') &
    (df_clean['image_name'].isin(busuclm_manual_exclusions))
)

df_clean = df_clean[~busuclm_manual_mask].copy()
print(f'BUS-UCLM count after manual Doppler correction: '
      f"{(df_clean['dataset'] == 'BUS-UCLM').sum()}")

### HESN Patient Excluded

All 13 images from patient HESN (`HESN_000.png`-`HESN_012.png`) show elevated colour saturation across genuine tissue regions, causing colour-based annotation detection to flag tissue as annotation overlay. Confirmed across the full series.

Not attributable to image mode: BrEaST is universally RGBA and shows no equivalent effect. Excluded at the patient level.

In [ ]:
hesn_exclusions = {f'HESN_{i:03d}.png' for i in range(13)}
print(f'HESN patient exclusion set: {len(hesn_exclusions)} images')
print(sorted(hesn_exclusions))

already_excluded = []
never_in_audit = []
for name in sorted(hesn_exclusions):
  # Check if already removed by an earlier step
  in_df_clean = name in df_clean.loc[df_clean['dataset'] == 'BUS-UCLM', 'image_name'].values
  in_audit = name in df.loc[df['dataset'] == 'BUS-UCLM', 'image_name'].values
  if not in_df_clean and not in_audit:
    never_in_audit.append(name)
  elif not in_df_clean and in_audit:
    already_excluded.append(name)

if never_in_audit:
  print(f'WARNING: {len(never_in_audit)} HESN filename(s) never found in audit master -> check naming: {never_in_audit}.')
if already_excluded:
  print(f'{len(already_excluded)} HESN image(s) already excluded by an earlier step (expected): {already_excluded}')

hesn_mask = (
    (df_clean['dataset'] == 'BUS-UCLM') &
    (df_clean['image_name'].isin(hesn_exclusions))
)

n_before = (df_clean['dataset'] == 'BUS-UCLM').sum()
found_count = hesn_mask.sum()
df_clean = df_clean[~hesn_mask].copy()
n_after = (df_clean['dataset'] == 'BUS-UCLM').sum()

print(f'\nBUS-UCLM images matching HESN exclusion set: {found_count}')
print(f'BUS-UCLM images before: {n_before}')
print(f'BUS-UCLM images after: {n_after}')

expected_removed = len(hesn_exclusions) - len(already_excluded)
if found_count != expected_removed:
  print(f'\nWARNING: expected to remove {expected_removed} '
        f'(13 total minus {len(already_excluded)} already removed), '
        f'but removed {found_count}. Investigate before proceeding.')
else:
  print(f'\nConfirmed: all 13 HESN images accounted for '
        f'({found_count} removed here, {len(already_excluded)} already excluded earlier).')

## 8. Checkpoint: Saving `df_clean` Before Inpainting

The persistence of `df_clean` to disk ensures against potential runtime disconnection during inpainting process.

This file represents the fully exclusion-applied, but not yet artefact-processed, state of the dataset.

In [ ]:
# Guard against accidentally saving a mask/segmentation file into df_clean
mask_pattern_words = ['mask', 'tumor', 'other']
pattern = '|'.join(mask_pattern_words)

suspicious = df_clean[
    df_clean['image_name'].str.lower().str.contains(pattern, regex=True)
]

if len(suspicious) > 0:
  print(f'WARNING: {len(suspicious)} suspicious filenames found.')
  print(suspicious[['image_name', 'dataset', 'filepath']].to_string())
  raise ValueError(
      'Mask-pattern filenames detected in df_clean. '
      'Investigate before proceeding.'
  )
else:
  print(f'Verified: no mask/tumor/other pattern filenames '
        f'in df_clean ({len(df_clean)} images checked).')

In [ ]:
checkpoint_path = RESULTS / 'df_clean_post_exclusion.csv'

df_clean.to_csv(checkpoint_path, index=False)

print(f'Checkpoint saved: {checkpoint_path}')
print(f'Rows: {len(df_clean)}')
print(f'Columns: {df_clean.columns.tolist()}')
print()
print('Dataset counts at checkpoint: ')
print(df_clean.groupby(['dataset', 'class'])['image_name'].count().to_string())

In [ ]:
df_clean = pd.read_csv(RESULTS / 'df_clean_post_exclusion.csv')

print(f'Loaded from checkpoint: {len(df_clean)} images')
print(df_clean.groupby(['dataset', 'class'])['image_name'].count().to_string())

## 9. Inpainting

The inpainting process is split into mask detection, creation and inpainting.

The datasets following the inpainting process are BUSI and BUS-UCLM, considering they are the datasets with artefacts present.

NB: `PROCESSED/` holds artefact-handled output at native resolution -- masking and inpainting applied, but not yet in final training format. Format standardisation happens in Section 10, producing `FINAL/`.

### 9.1 BUSClean Setup

BUSClean (Bunnell et al., PLOS One 2024; MIT licensed) is adopted for artefact detection. The repository has no tagged releases, so the clone is pinned to a specific commit for reproducibility. Used for detection only; cropping functionality is not used.

In [ ]:
REPO_DIR = '/content/bus-cleaning'
PINNED_COMMIT = 'd2630a435cd257e366a516a3c5c5cf26c27b3243'

if not os.path.exists(REPO_DIR):
  !git clone https://github.com/hawaii-ai/bus-cleaning.git {REPO_DIR}

%cd {REPO_DIR}
!git checkout {PINNED_COMMIT}
!pip install .

current_commit = !git rev-parse HEAD
assert current_commit[0] == PINNED_COMMIT, (
    f'Checked out {current_commit[0]}, expected {PINNED_COMMIT}. '
    'Investigate before proceeding.'
)
print(f'Confirmed: bus-cleaning pinned to commit {PINNED_COMMIT}')

In [ ]:
sys.path.append('/content/bus-cleaning')

### 9.2 Calliper Detection (BUSI)

`enhance_image_full_frame` and `detect_callipers`, adapted from BUSClean's `enhance_image` and `detect_anno_BUSI` (`modules/artifacts.py`).

Modified BUSClean's method with Hough-line fallback produced false negative detections, leading to retention of annotations through the full mask pipeline.

The current version treats BUSClean's enhanced edge-map directly as the mask, with no assumption about calliper shape. Connected-component filtering removes small isolated noise before dilation closes gaps in dashed/dotted lines.

Returns a binary mask, not boxes.

In [ ]:
artefact_detection_code = '''
from itertools import combinations
import cv2
import numpy as np
from PIL import Image, ImageFilter
from modules.artifacts import detect_anno


def enhance_image_full_frame(im: Image.Image, show_thresh: bool = False) -> Image.Image:
  """
  Modified version of BUSClean's enhance_image (modules/artifacts.py).
  Original: MIT licensed, hawaii-ai/bus-cleaning, commit d2630a4.

  CHANGE from original: runs FIND_EDGES on the full, uncropped frame
  instead of blacking out a 15% border: lesions near BUSI's edge were being discarded.

  Args:
    im (Image.Image): RGB PIL image.
    show_thresh (bool): If True, show debugging prints/images.

  Returns:
    Image.Image: the enhanced (edge-map) image.
  """
  dilate = 1
  new_image = im.filter(ImageFilter.FIND_EDGES)
  if show_thresh:
    display(new_image)

  array_im: np.ndarray = np.array(new_image)
  array_im = cv2.cvtColor(array_im, cv2.COLOR_RGB2GRAY)
  _, binary_image = cv2.threshold(array_im, 210, 255, cv2.THRESH_BINARY)
  enhance = (Image.fromarray(binary_image)).filter(ImageFilter.MaxFilter)

  enhance = np.array(enhance)
  for i in range(dilate):
    enhance = cv2.dilate(enhance, np.ones((2, 2), np.uint8))
  enhance = Image.fromarray(enhance)

  if show_thresh:
    display(enhance)

  return enhance


def detect_callipers(im, show_thresh: bool = False,
                     dilation_size: int = 2, min_component_area: int = 22):
  """
  Modified version of BUSClean's detect_anno_BUSI (modules/artifacts.py).
  Original: MIT licensed, hawaii-ai/bus-cleaning, commit d2630a4.

  Treats the enhanced edge-map as the mask directly, dilated to close gaps in
  dashed/dotted lines. Connected-component filtering removes anything below
  min_component_area before dilation. No assumption is made about calliper shape,
  due to high heterogeneity of artefacts in BUSI.

  Args:
    im: the enhanced (edge-map) image, as returned by
        enhance_image_full_frame().
    show_thresh (bool): If True, show debugging prints/images.
    dilation_size (int): closes gaps in dashed/dotted annotation lines.
    min_component_area (int): pixel area below which is a connected component is
                              treated as noise, not a real annotation.

  Returns:
    np.ndarray: binary mask (uint8), same size as im, 255 = flagged.
  """
  arr = np.array(im)

  num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(arr, connectivity=8)
  filtered = np.zeros_like(arr)
  for label_id in range(1, num_labels):
    if stats[label_id, cv2.CC_STAT_AREA] >= min_component_area:
      filtered[labels == label_id] = 255

  kernel = np.ones((dilation_size, dilation_size), np.uint8)
  mask = cv2.dilate(filtered, kernel, iterations=1)

  if show_thresh:
    display(Image.fromarray(mask))
  return mask
'''

output_path = SRC / 'artefact_detection.py'
output_path.write_text(artefact_detection_code)
print(f'Saved: {output_path}')

### 9.3 Text Detection

`get_text_attributes_batch`, adapted from BUSClean's `get_text_attributes` (`modules/text.py`) uses a single shared EasyOCR reader instance across all calls. Returns bounding boxes and raw OCR text only. No text classification into semantic categories is applied.

`paragraph=True` (BUSClean's original setting) was found to merge spatially separate real detections into one bounding box spanning real tissue between them.

Two filters guard against OCR hallucinating text on noisy or textured tissue: `max_box_fraction` rejects any detection spanning an implausibly large fraction of the frame, and `min_confidence` rejects low-condfidence reads. Confidence alone was tested and found insufficient on its own, so genuine annotations dropped by this threshold are expected to still be caught by calliper/edge detection.

In [ ]:
text_detection_code = r'''
import re
import cv2
import numpy as np
from PIL import Image

def get_text_attributes_batch(filename: str, reader, show_thresh: bool = False,
                              max_box_fraction: float = 0.25, min_confidence = 0.7):
  """
  Modified version of BUSClean\\'s get_text_attributes (modules/text.py).
  Original: MIT licensed, hawaii-ai/bus-cleaning, commit d2630a4.

  CHANGES from original:
    1. reader is passed in by the caller (created once, outside any loop) instead
       of instantiated fresh inside every cell.
    2. The call to text_helper() and everything it calls is removed entirely.
       text_helper classifies OCR text into clinical semantic categories for BUSClean\\'s
       own knowledge-extraction use case; it plays no role in producing the bounding
       boxes used here for annotation removal.
    3. paragraph=True changed to paragraph=False. Paragraph mode\\'s purpose was
       producing reading-order merged text for semantic classification, which, when
       left as is, merged spatially separate real detections into one bounding box
       spanning tissue between them.
    4. max_box_fraction: rejects any box whose width or height exceeds this fraction
       of the image's shorter dimension, considering the permissive threshold misread
       a textured/hypoechoic tissue region as a large block of text.
    5. min_confidence: paragraph=False restores per-detection confidence scores,
       enabling direct confidence filtering as a more principled backstop against
       the same hallucination issue.

  Args:
    filename (str): path to the image file.
    reader (easyocr.Reader): a single, already-instantiated EasyOCR reader, created
                             once by the caller and reused across calls.
    show_thresh (bool): If True, show prints/images for debugging.
    max_box_fraction (float): reject boxes exceeding this fraction of the image\\'s
                              shorter dimension.
    min_confidence (float): reject detections below this confidence.

  Returns:
    bounding_box_coord (list): 4-corner boxes for each detected text region passing
                               all filters.
    orig_text (str): all raw OCR text found, concatenated, for logging.
  """
  img = Image.open(filename).convert('RGB')
  width, height = img.size
  im = np.array(img)

  for x in range(0, 70):
    im[0] = 0
    appending_black = [im[0]]
    im = np.append(im, appending_black, axis=0)

  orig_text = []
  bounding_box_coord = []
  max_dim = max_box_fraction * min(width, height)

  full_text = reader.readtext(im, x_ths=2.4, paragraph=False, mag_ratio=2.1, low_text=0.2, link_threshold=0.35)

  for bbox, text, confidence in full_text:
    bbox = np.asarray(bbox, dtype='int')
    orig_text.append(text)

    box_w = bbox[2][0] - bbox[0][0]
    box_h = bbox[2][1] - bbox[0][1]

    if (bbox[2][1] > height/8 and len(text) > 2 and re.match(r'^\W+$', " ".join(text.split())) is None
        and box_w <= max_dim and box_h <= max_dim and confidence >= min_confidence):
      if bbox[2][1] > height:
        bbox[2][1] = height
      bounding_box_coord.append([bbox[0], bbox[1], bbox[2], bbox[3]])

      if show_thresh:
        cv2.rectangle(im, bbox[0], bbox[2], (255, 255, 255), 2)

    if show_thresh:
      cv2.circle(im, bbox[0], 5, (255, 255, 255), 2)
      cv2.circle(im, bbox[2], 5, (255, 255, 255), 2)

  orig_text = ' '.join(orig_text)

  if show_thresh:
    display(Image.fromarray(im))

  return bounding_box_coord, orig_text
'''

output_path = SRC / 'text_detection.py'
output_path.write_text(text_detection_code)
print(f'Saved: {output_path}')

### 9.4 Colour-Based Annotation Detection

`detect_colour_annotations` and `detect_colour_annotations_by_range`. HSV threshold ranges for green annotations (arrows, callipers, ROI outliers), yellow annotations (scanner logo, callipers), and the BI-RADS digram (in BUS-UCLM dataset).

Each range is dilated independently with an elliptical kernel (3px for annotations, 6px for the diagram) and unioned. Detection is contour-preserving throughout; no bounding-box expansion is applied to any range.

In [ ]:
colour_detection_code = r'''
import cv2
import numpy as np
from PIL import Image

DEFAULT_COLOUR_RANGES = {
  'green_annotation': ((33, 45, 65), (92, 255, 255)),
  'yellow_annotation': ((16, 50, 70), (38, 255, 255)),
  'diagram_blue_purple': ((90, 40, 70), (162, 255, 255)),
}

DEFAULT_KERNEL_SIZES = {
  'green_annotation': 3,
  'yellow_annotation': 3,
  'diagram_blue_purple': 6,
}

def detect_colour_annotations_by_range(im, colour_ranges=None, dilate_iterations=1, kernel_sizes=None):
  """
  Detects each colour range separately, each dilated by its own elliptical kernel
  size.

  Args:
    im (Image.Image): RGB PIL image.
    colour_ranges (dict or None): {name: (lower_hsv, upper_hsv)}. None value uses
                                  DEFAULT_COLOUR_RANGES.
    dilate_iterations (int): growth applied to each mask.
    kernel_sizes (dict or None): {name: kernel_size}. None value uses DEFAULT_KERNEL_SIZES.

  Returns:
    dict: {range_name: uint8 mask}, same (height, width) as im.
  """
  array_im = np.array(im)
  img_hsv = cv2.cvtColor(array_im, cv2.COLOR_RGB2HSV)

  if colour_ranges is None:
    colour_ranges = DEFAULT_COLOUR_RANGES
  if kernel_sizes is None:
    kernel_sizes = DEFAULT_KERNEL_SIZES

  masks = {}
  for name, (lower, upper) in colour_ranges.items():
    match = cv2.inRange(img_hsv, np.array(lower), np.array(upper))
    if dilate_iterations > 0:
      size = kernel_sizes.get(name, 3)
      kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))
      match = cv2.dilate(match, kernel, iterations=dilate_iterations)
    masks[name] = match

  return masks


def detect_colour_annotations(im, colour_ranges=None, dilate_iterations=1, kernel_sizes=None):
  """
  Union of all per-range masks. Used as the default colour_fn for build_final_mask.
  """
  masks = detect_colour_annotations_by_range(im, colour_ranges, dilate_iterations, kernel_sizes)
  height, width = np.array(im).shape[:2]
  combined = np.zeros((height, width), dtype=np.uint8)
  for m in masks.values():
    combined = cv2.bitwise_or(combined, m)
  return combined
'''

output_path = SRC / 'colour_detection.py'
output_path.write_text(colour_detection_code)
print(f'Saved: {output_path}')

### 9.5 Combined Mask Generation

`build_final_mask` unions calliper detection, text detection, and colour-based annotation detection into a single binary removal mask per image.

In [ ]:
combine_masks_code = r'''
import cv2
import numpy as np
from PIL import Image
from colour_detection import detect_colour_annotations

def boxes_to_mask(boxes, image_shape):
  """
  Rasterizes a list of 4-corner boxes (as returned by detect_callipers()
  and get_text_attributes_batch()) into a single binary mask.
  """
  mask = np.zeros(image_shape[:2], dtype=np.uint8)
  for box in boxes:
    pts = np.array(box, dtype=np.int32).reshape((-1, 1, 2))
    cv2.fillPoly(mask, [pts], 255)
  return mask


def build_final_mask(image_path, reader, enhance_fn, calliper_fn, text_fn,
                      colour_fn=detect_colour_annotations, colour_ranges=None,
                      colour_fn_kwargs=None, show_thresh=False):
  """
  Union of calliper detection, text detection and colour-annotation detection.

  NOTE: calliper_fn now returns a mask array directly, considering detect_callipers
  was rewritten to use the edge-map directly rather than shape-matched boxes.

  Args:
    image_path (str): path to the image file.
    reader (easyocr.Reader): shared, pre-instantiated OCR reader.
    enhance_fn, calliper_fn, text_fn, colour_fn: detection functions.
    colour_ranges (dict or None): passed through to colour_fn
    colour_fn_kwargs (dict or None): extra keyword arguments forwarded to colour_fn.
    show_thresh (bool): debug visualisation passthrough.

  Returns:
  np.ndarray: final uint8 binary mask, 255 = remove/inpaint, 0 = leave untouched.
  """
  img = Image.open(image_path).convert('RGB')
  image_shape = np.array(img).shape

  enhanced = enhance_fn(img, show_thresh=False)
  calliper_mask = calliper_fn(enhanced, show_thresh=show_thresh)

  text_boxes, _ = text_fn(image_path, reader, show_thresh=show_thresh)
  text_mask = boxes_to_mask(text_boxes, image_shape)

  # Calliper and text detections are both boxes -> rasterize both into one mask
  box_mask = cv2.bitwise_or(calliper_mask, text_mask)

  colour_fn_kwargs = colour_fn_kwargs or {}
  colour_mask = colour_fn(img, colour_ranges=colour_ranges, **colour_fn_kwargs)

  # Final mask is the union of box-based and colour-based detections
  return cv2.bitwise_or(box_mask, colour_mask)
'''

output_path = SRC / 'mask_utils.py'
output_path.write_text(combine_masks_code)
print(f'Saved: {output_path}')

In [ ]:
import sys

for path in [str(SRC), '/content/bus-cleaning']:
  if path not in sys.path:
    sys.path.append(path)

from artefact_detection import enhance_image_full_frame, detect_callipers
from text_detection import get_text_attributes_batch
from colour_detection import detect_colour_annotations, detect_colour_annotations_by_range
from mask_utils import boxes_to_mask, build_final_mask

print('Imports successful.')

### 9.6 Batch Mask Generation

`build_final_mask` is run over every BUSI and BUS-UCLM image in `df_clean`, saving each binary mask and a translucent red view overlay to disk. Progress and partial results are checkpointed every 50 images, independent of whether individual images succeed or fail, so a runtime doesn't lose completed work.

In [ ]:
import easyocr

# Reuse the existing reader if it exists, otherwise create a new one
try:
  reader
  print('Using existing reader instance.')
except NameError:
  reader = easyocr.Reader(['en'], gpu=True)
  print('Created new reader instance.')

MASKS_DIR = OUTPUTS / 'masks'
REVIEW_DIR = OUTPUTS / 'masks_review'
for d in [MASKS_DIR / 'BUSI', MASKS_DIR / 'BUS-UCLM',
          REVIEW_DIR / 'BUSI', REVIEW_DIR / 'BUS-UCLM']:
  d.mkdir(parents=True, exist_ok=True)

def make_overlay(image_path, mask, alpha=0.45):
  """Translucent red highlight over flagged mask regions, for fast visual review."""
  img = np.array(Image.open(image_path).convert('RGB'))
  highlighted = img.copy()
  highlighted[mask > 0] = [255, 0, 0]
  blended = cv2.addWeighted(highlighted, alpha, img, 1 - alpha, 0)
  return Image.fromarray(blended)

df_clean = pd.read_csv(RESULTS / 'df_clean_post_exclusion.csv')
to_process = df_clean[df_clean['dataset'].isin(['BUSI', 'BUS-UCLM'])].reset_index(drop=True)
print(f'Image to process: {len(to_process)}')

In [ ]:
mask_log = []
failures = []

for idx, row in to_process.iterrows():
  try:
    mask = build_final_mask(
        row['filepath'], reader,
        enhance_fn=enhance_image_full_frame,
        calliper_fn=detect_callipers,
        text_fn=get_text_attributes_batch,
        show_thresh=False
    )

    mask_subdir = MASKS_DIR / row['dataset']
    review_subdir = REVIEW_DIR / row['dataset']

    Image.fromarray(mask).save(mask_subdir / row['image_name'])
    make_overlay(row['filepath'], mask).save(review_subdir / row['image_name'])

    mask_log.append({
        'image_name': row['image_name'],
        'dataset': row['dataset'],
        'class': row['class'],
        'pixels_flagged': int((mask > 0).sum()),
        'pct_flagged': round(100 * (mask > 0).sum() / mask.size, 2)
    })

  except Exception as e:
    failures.append({
        'image_name': row['image_name'],
        'dataset': row['dataset'],
        'error': str(e)
    })

  if(idx + 1) % 50 == 0:
    pd.DataFrame(mask_log).to_csv(RESULTS / 'mask_generation_log_partial.csv', index=False)
    print(f'  ...{idx + 1}/{len(to_process)} processed')

mask_log_df = pd.DataFrame(mask_log)
mask_log_df.to_csv(RESULTS / 'mask_generation_log.csv', index=False)

print(f'\nDone. Processed: {len(to_process)}')
print(f'Successful: {len(mask_log)}')
print(f'Failed: {len(failures)}')

if failures:
  pd.DataFrame(failures).to_csv(RESULTS / 'mask_generation_failures.csv', index=False)
  print('\nFailures (saved to mask_generation_failures.csv): ')
  for f in failures:
    print(f"  {f['image_name']} ({f['dataset']}): {f['error']}")
else:
  print('No failures.')

print('\nCoverage by dataset: ')
print(mask_log_df.groupby('dataset')['image_name'].count().to_string())

print('\nFlagged-pixel distribution (sanity check for outliers): ')
print(mask_log_df.groupby('dataset')['pct_flagged'].describe().to_string())

### 9.7 Validation Against Known Failure Cases

Manual review during development surfaced three failure modes, each fixed: a missed asymmetric calliper, OCR hallucinating text over a lesion, and OCR merging separate real detections into one bounding box spanning a lesion.

Full pipeline is run against each failure mode, plus a stratified random sample and a worst-case-by-coverage check, as the validation evidence for mask generation.

In [ ]:
from mask_utils import boxes_to_mask

def diagnose_image(filename, dataset_hint=None):
  """
  Runs every detection mechanism on one named image and displays each stage
  separately, for regression-testing known failure cases and isolating which
  mechanism is responsible when something looks wrong.
  """
  query = df_clean['image_name'] == filename
  if dataset_hint:
    query &= df_clean['dataset'] == dataset_hint
  matches = df_clean[query]
  if len(matches) == 0:
    raise ValueError(f'{filename} not found in df_clean.')
  if len(matches) > 1:
    print(f'WARNING: {len(matches)} rows match -- using the first.')
  r = matches.iloc[0]

  print(f"=== {r['image_name']} ({r['dataset']}, {r['class']}) ===")
  img = Image.open(r['filepath']).convert('RGB')
  im_arr = np.array(img)
  display(img)

  enhanced = enhance_image_full_frame(img, show_thresh=False)
  calliper_mask = detect_callipers(enhanced, show_thresh=False)
  print(f'Calliper mask: {(calliper_mask > 0).sum()} px flagged')

  ocr_results = reader.readtext(im_arr, x_ths=2.4, paragraph=False, mag_ratio=2.1,
                                low_text=0.2, link_threshold=0.35)

  print('\nRaw OCR detections (confidence, calliper-mask overlap):')
  for bbox, text, conf in sorted(ocr_results, key=lambda x: x[2]):
    bbox_arr = np.asarray(bbox, dtype='int')
    x1, x2 = bbox_arr[:, 0].min(), bbox_arr[:, 0].max()
    y1, y2 = bbox_arr[:, 1].min(), bbox_arr[:, 1].max()
    region = calliper_mask[y1:y2, x1:x2]
    overlap = (region > 0).sum() / max((x2 - x1) * (y2 -y1), 1)
    kept = conf >= 0.7
    print(f'  "{text}"  conf={conf:.3f}   overlap={overlap:.1%} '
          f'{"KEPT" if kept else "dropped (below min_confidence)"}')

  final_mask = build_final_mask(
      r['filepath'], reader,
      enhance_fn=enhance_image_full_frame,
      calliper_fn=detect_callipers,
      text_fn=get_text_attributes_batch,
      colour_fn=detect_colour_annotations,
      show_thresh=False
  )

  print(f'\nFinal union mask: {(final_mask > 0).sum()} px flagged')
  display(make_overlay(r['filepath'], final_mask))

known_failure_cases = [
    'benign (200).png',
    'MENE_002.png',
    'DAPA_020.png',
]

for fname in known_failure_cases:
  diagnose_image(fname)
  print('\n' + '='*70 + '\n')

In [ ]:
# Stratified random sampling
review_sample = to_process.groupby(['dataset', 'class'], group_keys=False).apply(
    lambda x: x.sample(min(len(x), 3), random_state=60)
)
print(review_sample[['image_name', 'dataset', 'class']].to_string())

for _, r in review_sample.iterrows():
    final_mask = build_final_mask(
        r['filepath'], reader,
        enhance_fn=enhance_image_full_frame,
        calliper_fn=detect_callipers,
        text_fn=get_text_attributes_batch,
        colour_fn=detect_colour_annotations,
        show_thresh=False
    )
    print(f"\n{r['image_name']} ({r['dataset']}, {r['class']})")
    display(make_overlay(r['filepath'], final_mask))

In [ ]:
# Worst-case check: images with the highest flagged-pixel percentage,
# post-batch-run.
mask_log_df = pd.read_csv(RESULTS / 'mask_generation_log.csv')
print(f'Images processed: {len(mask_log_df)}')
print('\nTop 10 by percent of image flagged:')
print(mask_log_df.sort_values('pct_flagged', ascending=False).head(10).to_string())

#### Table Figure: Flagged-Pixel Distribution

Checks the mean, standard deviation, minimum and maximum values, as well as 25th, 50th, and 75th percentile for masked pixel distribution across `BUSI` and `BUS-UCLM` datasets per class.

In [ ]:
# Variations in flagged-pixel distribution per class
print(mask_log_df.groupby(['dataset', 'class'])['pct_flagged'].describe().to_string())

#### Illustrative Figure: OCR Confidence Filtering

Visualises raw EasyOCR detections on a chosen image, colour-codeed by whether each detection cleared `min_confidence=0.7` -- green kept, red dropped.

_Used to illustrate the confidence-based filtering decision in the methodology chapter._

In [ ]:
def visualise_ocr_confidence(filenames, dataset_hint=None, confidence_threshold=0.7,
                             save_dir=None):
  """
  Draws raw EasyOCR detections on one or more named images, colour-coded by whether
  each cleared min_confidence: green kept, red dropped.
  Built for illustrative thesis figures, not regression testing.

  Args:
    filenames (str or list[str]): a single filename, or a list to process in turn.
    save_dir (Path or None): if given, saves each figure as '<filename_stem>_ocr_confidence.png'.

  Returns:
  dict: {filename: PIL.Image}, one entry per image processed.
  """
  if isinstance(filenames, str):
    filenames = [filenames]

  results_out = {}
  for filename in filenames:
    query = df_clean['image_name'] == filename
    if dataset_hint:
      query &= df_clean['dataset'] == dataset_hint
    matches = df_clean[query]
    if len(matches) == 0:
      print(f'WARNING: {filename} not found in df_clean -- skipped.')
      continue
    r = matches.iloc[0]

    img = Image.open(r['filepath']).convert('RGB')
    im_arr = np.array(img)

    ocr_results = reader.readtext(im_arr, x_ths=2.4, paragraph=False, mag_ratio=2.1,
                                  low_text=0.2, link_threshold=0.35)
    print(f"{r['image_name']} ({r['dataset']}, {r['class']}) -- {len(ocr_results)} detection(s)")

    vis = im_arr.copy()
    for bbox, text, conf in ocr_results:
      bbox = np.asarray(bbox, dtype='int')
      color = (0, 255, 0) if conf >= confidence_threshold else (255, 0, 0)
      cv2.rectangle(vis, tuple(bbox[0]), tuple(bbox[2]), color, 2)
      cv2.putText(vis, f'"{text}" {conf:.2f}', (bbox[0][0], max(bbox[0][1] - 5, 10)),
                  cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
      print(f'  "{text}"  confidence={conf:.3f}  '
            f'{"KEPT" if conf >= confidence_threshold else "dropped"}')

    result_img = Image.fromarray(vis)
    display(result_img)

    if save_dir:
      save_dir = Path(save_dir)
      save_dir.mkdir(parents=True, exist_ok=True)
      save_path = Path(save_dir) / f'{Path(filename).stem}_ocr_confidence.png'
      result_img.save(save_path)
      print(f'Saved: {save_path}')

    results_out[filename] = result_img
    print()

  return results_out

In [ ]:
visualise_ocr_confidence(known_failure_cases, save_dir= OUTPUTS / 'figures_ocr')

#### Illustrative Figure: Mask Overlay

Overlays the original image and its saved mask for a chosen image.

_Used to illustrate the match between the generated mask and the original image._

In [ ]:
def show_mask_overlay(filenames, dataset_hint=None, save_dir=None):
  """
  Loads and displays the original image alongside its saved mask overlay from
  REVIEW_DIR. Requires the batch mask-generation loop (9.6) to have already run
  for these images.
  """
  if isinstance(filenames, str):
    filenames = [filenames]

  results_out = {}
  for filename in filenames:
    query = df_clean['image_name'] == filename
    if dataset_hint:
      query &= df_clean['dataset'] == dataset_hint
    matches = df_clean[query]
    if len(matches) == 0:
      print(f'WARNING: {filename} not found in df_clean -- skipped.')
      continue
    r = matches.iloc[0]

    overlay_path = REVIEW_DIR / r['dataset'] / r['image_name']
    if not overlay_path.exists():
      print(f'WARNING: no saved overlay at {overlay_path} -- '
            'has the 9.6 batch loop run for this image yet? Skipped.')
      continue

    print(f"{r['image_name']} ({r['dataset']}, {r['class']})")
    print('Original:')
    display(Image.open(r['filepath']).convert('RGB'))
    print('Mask overlay (loaded from disk, not recomputed):')
    overlay = Image.open(overlay_path)
    display(overlay)

    if save_dir:
      save_dir = Path(save_dir)
      save_dir.mkdir(parents=True, exist_ok=True)
      save_path = Path(save_dir) / f'{Path(filename).stem}_mask_overlay.png'
      overlay.save(save_path)
      print(f'Saved: {save_path}')

    results_out[filename] = overlay
    print()

  return results_out

In [ ]:
show_mask_overlay(known_failure_cases, save_dir=OUTPUTS / 'figures_mask')

### 9.8 Inpainting Radius Selection

`inpaintRadius=10`, `cv2.INPAINT_NS`, selected after comparing radii 3, 5, and 10 and comparing NS directly against TELEA on a set of images that have shown the most annotations present. NS at radius 10 produced a cleaner reconstruction than TELEA on this case. Used uniformly across BUSI and BUS-UCLM.



In [ ]:
test_images = [
    {'filename': 'benign (102).png', 'dataset': 'BUSI'},
    {'filename': 'ALWI_019.png', 'dataset': 'BUS-UCLM'},
    {'filename': 'ALWI_000.png', 'dataset': 'BUS-UCLM'},
]

radii = [3, 5, 10]
algorithms = [('NS', cv2.INPAINT_NS), ('TELEA', cv2.INPAINT_TELEA)]

for test in test_images:
  filename = test['filename']
  dataset = test['dataset']

  row_r = df_clean[(df_clean['dataset'] == dataset) & (df_clean['image_name'] == filename)].iloc[0]
  img_arr = np.array(Image.open(row_r['filepath']).convert('RGB'))
  mask = np.array(Image.open(MASKS_DIR / dataset / filename))

  print(f"=== {filename}  ({dataset}, {row_r['class']}) -- {int((mask > 0).sum())} px flagged ===")
  print('Original:')
  display(Image.fromarray(img_arr))

  for radius in radii:
    for algo_name, algo_flag in algorithms:
      result = cv2.inpaint(img_arr, mask, inpaintRadius=radius, flags=algo_flag)
      print(f'{algo_name}, radius={radius}:')
      display(Image.fromarray(result))

### 9.9 Mask Check

Checks whether every image has its own mask generated by previous steps, before inpainting.

In [ ]:
missing_masks = []

for _, r in to_process.iterrows():
  mask_path = MASKS_DIR / r['dataset'] / r['image_name']
  if not mask_path.exists():
    missing_masks.append({'image_name': r['image_name'], 'dataset': r['dataset']})

if missing_masks:
  print(f'WARNING: {len(missing_masks)} images missing a saved mask file:')
  print(pd.DataFrame(missing_masks).to_string())
  raise ValueError('Missing mask files; investigate before proceeding.')
else:
  print(f'Verified: all {len(to_process)} images (BUSI + BUS-UCLM) have a saved mask.')

### 9.10 Ablation Arm: Annotated (BUSI, Untouched)

Copies the post-exclusion BUSI images as-is into `BUSI_annotated`, organized by class. This is the ablation baseline — no masking, no inpainting — using `shutil.copy2` to preserve the original file bytes exactly, rather than an open/re-save round trip.

In [ ]:
import shutil

busi_df = df_clean[df_clean['dataset'] == 'BUSI']

for cls in busi_df['class'].unique():
  (BUSI_PROCESSED / 'BUSI_annotated' / cls).mkdir(parents=True, exist_ok=True)

copied = 0
copy_failures = []

for _, r in busi_df.iterrows():
  try:
    dest = BUSI_PROCESSED / 'BUSI_annotated' / r['class'] / r['image_name']
    shutil.copy2(r['filepath'], dest)
    copied += 1
  except Exception as e:
    copy_failures.append({'image_name': r['image_name'], 'error': str(e)})

print(f'Copied {copied}/{len(busi_df)} images to BUSI_annotated.')
if copy_failures:
  print(f'Failures: {len(copy_failures)}')
  print(pd.DataFrame(copy_failures).to_string())

### 9.11 NS Inpainting (BUSI + BUS-UCLM)

Runs `cv2.inpaint` (NS, radius=10 — see 9.8) over every image using its previously saved mask — BUSI into `BUSI_inpainted` (the main experimental arm), BUS-UCLM into its single processed path. Checkpoints progress every 50 images regardless of per-image success or failure.

In [ ]:
def inpaint_batch(subset_df, mask_source_dir, output_dir, radius=10, algorithm=cv2.INPAINT_NS):
  for cls in subset_df['class'].unique():
    (output_dir / cls.lower()).mkdir(parents=True, exist_ok=True)

  inpaint_log = []
  failures = []

  for idx, (_, r) in enumerate(subset_df.iterrows()):
    try:
      mask = np.array(Image.open(mask_source_dir / r['image_name']))
      img_arr = np.array(Image.open(r['filepath']).convert('RGB'))
      result = cv2.inpaint(img_arr, mask, inpaintRadius=radius, flags=algorithm)
      Image.fromarray(result).save(output_dir / r['class'].lower() / r['image_name'])
      inpaint_log.append({'image_name': r['image_name'], 'class': r['class'].lower()})
    except Exception as e:
      failures.append({'image_name': r['image_name'], 'error': str(e)})

    if (idx + 1) % 50 == 0:
      print(f'  ...{idx + 1}/{len(subset_df)} processed')

  print(f'\nDone. Processed: {len(subset_df)}   Successful: {len(inpaint_log)}    Failed: {len(failures)}')
  if failures:
    for f in failures:
      print(f"  {f['image_name']}: {f['error']}")
  return inpaint_log, failures

print('=== NS Inpainting: BUSI -> BUSI_inpainted ===')
busi_inpaint_log, busi_inpaint_failures = inpaint_batch(
    df_clean[df_clean['dataset'] == 'BUSI'],
    mask_source_dir=MASKS_DIR / 'BUSI',
    output_dir=BUSI_PROCESSED / 'BUSI_inpainted'
)

print('=== NS Inpainting: BUS-UCLM ===')
busuclm_inpaint_log, busuclm_inpaint_failures = inpaint_batch(
    df_clean[df_clean['dataset'] == 'BUS-UCLM'],
    mask_source_dir=MASKS_DIR / 'BUS-UCLM',
    output_dir=BUSUCLM_PROCESSED
)

### 9.12 Ablation Arm: Black-Box Masking (BUSI)

Zeroes out every masked region rather than reconstructing it — the third ablation arm, using the identical masks as `BUSI_inpainted` so the only variable between the two arms is reconstruction vs. deletion.

In [ ]:
def blackbox_image(img_array, mask):
  """Replaces every masked pixel with black. No blending, no reconstruction."""
  result = img_array.copy()
  result[mask > 0] = [0, 0, 0]
  return result

In [ ]:
def blackbox_batch(subset_df, mask_source_dir, output_dir):
  for cls in subset_df['class'].unique():
    (output_dir / cls.lower()).mkdir(parents=True, exist_ok=True)

  blackbox_log = []
  failures = []

  for idx, (_, r) in enumerate(subset_df.iterrows()):
    try:
      mask = np.array(Image.open(mask_source_dir / r['image_name']))
      img_arr = np.array(Image.open(r['filepath']).convert('RGB'))
      result = blackbox_image(img_arr, mask)
      Image.fromarray(result).save(output_dir / r['class'].lower() / r['image_name'])
      blackbox_log.append({'image_name': r['image_name'], 'class': r['class'].lower()})
    except Exception as e:
      failures.append({'image_name': r['image_name'], 'error': str(e)})

    if (idx + 1) % 50 == 0:
      print(f'  ...{idx + 1}/{len(subset_df)} processed')

  print(f'\nDone. Processed: {len(subset_df)}   Successful: {len(blackbox_log)}   Failed: {len(failures)}')

  if failures:
    for f in failures:
      print(f"  {f['image_name']}: {f['error']}")

  return blackbox_log, failures

print('=== Black-Box Masking: BUSI -> BUSI_masked ===')
busi_blackbox_log, busi_blackbox_failures = blackbox_batch(
    df_clean[df_clean['dataset'] == 'BUSI'],
    mask_source_dir=MASKS_DIR / 'BUSI',
    output_dir=BUSI_PROCESSED / 'BUSI_masked'
)

In [ ]:
print('Final counts per output folder:')
for label, path, expected in [
    ('BUSI_annotated', BUSI_PROCESSED / 'BUSI_annotated', len(df_clean[df_clean['dataset'] == 'BUSI'])),
    ('BUSI_inpainted', BUSI_PROCESSED / 'BUSI_inpainted', len(df_clean[df_clean['dataset'] == 'BUSI'])),
    ('BUSI_masked', BUSI_PROCESSED / 'BUSI_masked', len(df_clean[df_clean['dataset'] == 'BUSI'])),
    ('BUS-UCLM', BUSUCLM_PROCESSED, len(df_clean[df_clean['dataset'] == 'BUS-UCLM'])),
]:
  actual = sum(1 for _ in path.rglob('*.png'))
  status = 'OK' if actual == expected else 'MISMATCH'
  print(f'  {label}: {actual}/{expected} files  [{status}]')

## 10. Format Standardisation

Converts every arm of every dataset to a common final format: grayscale-equivalent content stored as 3-channel RGB, letterboxed (aspect-ratio-preserving padding, not stretched) to 224×224. Letterboxing uses black padding.

#### Supporting Checks

Evidence backing the decisions documented above: square image count/padding verification, BrEaST alpha channel/colour investigation that resolved the flagged inconsistencies.

In [ ]:
square_count = (df_clean['width'] == df_clean['height']).sum()
print(f'Images with width == height (no padding needed): {square_count} / {len(df_clean)}')

In [ ]:
assert {'width', 'height'}.issubset(df_clean.columns), 'width/height mssing from df_clean. Check upstream.'

def compute_padding_fraction(width, height, target_size=224):
  scale = target_size / max(width, height)
  new_w, new_h = width * scale, height * scale
  real_area = new_w * new_h
  return 1 - (real_area / (target_size * target_size))

df_clean['padding_fraction'] = df_clean.apply(
    lambda r: compute_padding_fraction(r['width'], r['height']), axis=1
)

print(df_clean.groupby(['dataset', 'class'])['padding_fraction'].describe().to_string())

In [ ]:
breast_df = df_clean[df_clean['dataset'] == 'BrEaST']

alpha_summary = []
for _, r in breast_df.iterrows():
  img = Image.open(r['filepath'])
  if img.mode =='RGBA':
    alpha = np.array(img.split()[-1])
    alpha_summary.append({
        'image_name': r['image_name'],
        'alpha_min': alpha.min(),
        'alpha_max': alpha.max(),
        'alpha_uniform': alpha.min() == alpha.max()
    })

alpha_df = pd.DataFrame(alpha_summary)
print(f'BrEaST images checked: {len(alpha_df)}')
print(f'Images with fully uniform alpha (safe to drop): {alpha_df["alpha_uniform"].sum()}')
print(f'Images with varying alpha (worth a closer look): {(~alpha_df["alpha_uniform"]).sum()}')

if (~alpha_df['alpha_uniform']).any():
  print('\nNon-uniform alpha images:')
  print(alpha_df[~alpha_df['alpha_uniform']].to_string())

In [ ]:
for _, r in alpha_df[~alpha_df['alpha_uniform']].iterrows():
  row_r = breast_df[breast_df['image_name'] == r['image_name']].iloc[0]
  img_arr = np.array(Image.open(row_r['filepath']))
  alpha = img_arr[:, :, 3]
  rgb_under_transparent = img_arr[:, :, :3][alpha == 0]

  print(f"=== {r['image_name']} ===")
  print(f'RGB mean under alpha=0 region: {rgb_under_transparent.mean(axis=0)}')
  print(f'RGB std under alpha=0 region: {rgb_under_transparent.std(axis=0)}')
  display(Image.fromarray(img_arr[:, :, :3]))

In [ ]:
breast_true_rgb = breast_df[breast_df['is_truly_rgb'] == True]

tinted_check = []
for _, r in breast_true_rgb.iterrows():
  img = Image.open(r['filepath']).convert('RGB')
  hsv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2HSV)
  pct_saturated = (hsv[:, :, 1] > 20).mean()
  tinted_check.append({'image_name': r['image_name'], 'pct_saturated': pct_saturated})

tinted_df = pd.DataFrame(tinted_check)
tinted_df['likely_tinted'] = tinted_df['pct_saturated'] > 0.5

merged = tinted_df.merge(alpha_df[['image_name', 'alpha_uniform']], on='image_name', how='left')
print(pd.crosstab(merged['likely_tinted'], merged['alpha_uniform']))

#### Check: Fan-Shaped Acquisition Geometry

Detects cone/fan-shaped acquisitions by checking whether both top corners of an image are predominantly black, which is the signature of curvilinear-probe geometry, as opposed to the rectangular field a linear probe produces. Checked across all three datasets.

In [ ]:
CORNER_FRAC = 0.08
BLACK_PIXEL_VALUE = 15
BLACK_FRACTION_THRESHOLD = 0.90

def detect_cone_shape(image_path,
                      corner_frac=CORNER_FRAC,
                      black_value=BLACK_PIXEL_VALUE,
                      black_fraction=BLACK_FRACTION_THRESHOLD):
  img = cv2.imread(str(image_path))
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
  h, w = gray.shape

  # Sample small patches from the top-left and top-right corners
  patch_w = int(w * corner_frac)
  patch_h = int(h * corner_frac)

  top_left = gray[0:patch_h, 0:patch_w]
  top_right = gray[0:patch_h, w - patch_w:w]

  # Fraction of near-black pixels in each corner patch
  tl_black_frac = (top_left < black_value).sum() / top_left.size
  tr_black_frac = (top_right < black_value).sum() / top_right.size

  # Both corners mostly black -> fan-shaped (cone) acquisition geometry
  is_cone = (tl_black_frac >= black_fraction) and (tr_black_frac >= black_fraction)

  return is_cone, round(tl_black_frac, 3), round(tr_black_frac, 3)

In [ ]:
cone_summary = []

for dataset_name in ['BUSI', 'BUS-UCLM', 'BrEaST']:
  subset = df_clean[df_clean['dataset'] == dataset_name]
  is_cone_flags = [detect_cone_shape(r['filepath'])[0] for _, r in subset.iterrows()]
  count = sum(is_cone_flags)
  cone_summary.append({
      'dataset': dataset_name,
      'cone_shaped': count,
      'total': len(subset),
      'pct': round(100 * count / len(subset), 1) if len(subset) > 0 else 0.0
  })

cone_summary_df = pd.DataFrame(cone_summary)
print(cone_summary_df.to_string(index=False))

#### Shared Conversion Functions



In [ ]:
def to_final_format(img_rgb, target_size=224):
  """
  RGB -> grayscale-equivalent (plain channel average)
  -> letterbox/resize -> replicate to 3-channel RGB.
  """
  gray = np.round(img_rgb.astype(np.float32).mean(axis=2)).astype(np.uint8)

  h, w = gray.shape
  scale = target_size / max(h, w)
  new_h, new_w = max(int(round(h * scale)), 1), max(int(round(w * scale)), 1)

  interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
  resized = cv2.resize(gray, (new_w, new_h), interpolation=interp)

  canvas = np.zeros((target_size, target_size), dtype=np.uint8)
  top = (target_size - new_h) // 2
  left = (target_size - new_w) // 2
  canvas[top:top + new_h, left:left + new_w] = resized

  final = cv2.cvtColor(canvas, cv2.COLOR_GRAY2RGB)
  assert final.shape == (target_size, target_size, 3), f'Unexpected shape: {final.shape}'
  assert final.dtype == np.uint8, f'Unexpected dtype: {final.dtype}'
  return final


def breast_rgba_to_rgb(img_rgba_array):
  """
  Explicitly zeroes RGB wherever alpha=0, then drops alpha, rather than trusting
  a default RGBA -> RGB conversion, which would silently preserve faint non-zero
  content confirmed present under transparency in 7 of BrEaST's 31 cone-shaped images.
  """
  rgb = img_rgba_array[:, :, :3].copy()
  alpha = img_rgba_array[:, :, 3]
  rgb[alpha == 0] = [0, 0, 0]
  return rgb


def verify_folder_writable(path):
  """
  Creates the folder if needed, then writes and immediately reads back a tiny test
  file to confirm the path is genuinely writable right now.
  """
  path.mkdir(parents=True, exist_ok=True)
  test_file = path / '.write_test'
  test_file.write_text('ok')
  confirmed = test_file.exists() and test_file.read_text() == 'ok'
  test_file.unlink()
  return confirmed


def finalize_batch(subset_df, output_dir, source_dir=None, filepath_col=None, is_breast=False):
  """
  Runs to_final_format over every image in subset_df.

  Args:
    - filepath_col: read from that df column directly (BrEaST raw source).
    - source_dir: read from source_dir / class / image_name (already-processed BUSI/BUS-UCLM arms).
    - is_breast: applies the explicit RGBA->RGB fix instead of a plain .convert('RGB')

  Note:
    Destination folder existence is re-checked immediately before each save,
    not only once up front because of the possible real propagation delay on
    Drive-mounted filesystems.
  """
  for cls in subset_df['class'].unique():
    folder = output_dir / cls.lower()
    if not verify_folder_writable(folder):
      raise RuntimeError(f'Could not confirm {folder} is writable. Investigate before proceeding.')
  print(f'Verified: all {len(subset_df["class"].unique())} destination folders writable.')

  log = []
  failures = []

  for idx, (_, r) in enumerate(subset_df.iterrows()):
    try:
      src_path = r[filepath_col] if filepath_col else (source_dir / r['class'].lower() / r['image_name'])

      if is_breast:
        img_raw = Image.open(src_path)
        img_arr = np.array(img_raw)
        img_rgb = breast_rgba_to_rgb(img_arr) if img_raw.mode == 'RGBA' else np.array(img_raw.convert('RGB'))
      else:
        img_rgb = np.array(Image.open(src_path).convert('RGB'))

      final = to_final_format(img_rgb)
      Image.fromarray(final).save(output_dir / r['class'].lower() / r['image_name'])

      log.append({'image_name': r['image_name'], 'class': r['class'].lower()})
    except Exception as e:
      failures.append({'image_name': r['image_name'], 'error': str(e)})

    if (idx + 1) % 50 == 0:
      print (f' ...{idx + 1}/{len(subset_df)} processed')

  print(f'\nDone. Processed: {len(subset_df)}   Successful: {len(log)}    Failed: {len(failures)}')
  if failures:
    for f in failures:
      print(f"  {f['image_name']}: {f['error']}")
  return log, failures

### 10.1 BUSI

Applying the defined functions to all three arms of BUSI dataset: inpainted, fully-annotated, and masked.

In [ ]:
busi_df = df_clean[df_clean['dataset'] == 'BUSI']

for arm in ['BUSI_annotated', 'BUSI_inpainted', 'BUSI_masked']:
  print(f'=== {arm} ===')
  log, fails = finalize_batch(
      busi_df,
      source_dir=BUSI_PROCESSED / arm,
      output_dir=BUSI_FINAL / arm
  )

### 10.2 BUS-UCLM

Applying the defined functions to the inpainted BUS-UCLM dataset.

In [ ]:
busuclm_df = df_clean[df_clean['dataset'] == 'BUS-UCLM']

print('=== BUS-UCLM ===')
busuclm_final_log, busuclm_final_fails = finalize_batch(
    busuclm_df,
    source_dir=BUSUCLM_PROCESSED,
    output_dir=BUSUCLM_FINAL
)

### 10.3 BrEaST

Applying the defined functions to the BrEaST dataset.

In [ ]:
breast_df = df_clean[df_clean['dataset'] == 'BrEaST']

print('=== BrEaST ===')
breast_final_log, breast_final_fails = finalize_batch(
    breast_df,
    filepath_col='filepath',
    output_dir=BREAST_FINAL,
    is_breast=True
)

#### Final Verification

In [ ]:
print('Final counts, Section 10 output:')
for label, path, expected in [
    ('BUSI_annotated', BUSI_FINAL / 'BUSI_annotated', len(busi_df)),
    ('BUSI_inpainted', BUSI_FINAL / 'BUSI_inpainted', len(busi_df)),
    ('BUSI_masked', BUSI_FINAL / 'BUSI_masked', len(busi_df)),
    ('BUS-UCLM', BUSUCLM_FINAL, len(busuclm_df)),
    ('BrEaST', BREAST_FINAL, len(breast_df),)
]:
  actual = sum(1 for _ in path.rglob('*.png'))
  status = 'OK' if actual == expected else 'MISMATCH'
  print(f'  {label}: {actual}/{expected} files [{status}]')